# 05. RAG vs агент

Сравниваем два режима на одном golden-наборе:

- **RAG** — один retrieval top-5 и одна генерация ответа;
- **агент** — LangGraph: сам решает, какие инструменты вызвать, затем синтезирует ответ.

Дизайн:

- 30 вопросов из `eval/golden/qa.jsonl`;
- одинаковый judge и рубрика 1–5 для обоих режимов;
- контекст RAG — top-5 чанков;
- контекст агента — наблюдения вызванных инструментов;
- отдельно 6 вопросов из `tool_qa.jsonl` для tool-choice метрик.

Главные метрики: качество ответа, latency, число итераций и корректность
выбора инструментов.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

repo: D:\diffeq-wiki-rag


In [2]:
import json
import time

import pandas as pd
from langgraph.checkpoint.memory import MemorySaver

from app.agent.graph import build_agent_graph
from app.agent.tools import set_pipeline_getter
from app.llm import get_llm
from app.rag.pipeline import build_rag_pipeline, format_context
from app.rag.store import build_vectorstore
from eval.agent_eval import run_agent
from eval.metrics import load_golden
from eval.rag_eval import citation_stats, judge_answer
from eval.tool_metrics import evaluate_tool_questions

items = load_golden()
tool_items = [
    json.loads(line)
    for line in Path("eval/golden/tool_qa.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

store = build_vectorstore()
pipeline = build_rag_pipeline(
    retriever=store.as_retriever(search_kwargs={"k": 5})
)
set_pipeline_getter(lambda: pipeline)
graph = build_agent_graph(checkpointer=MemorySaver())
llm = get_llm()

print("вопросов:", len(items))
print("tool-вопросов:", len(tool_items))

D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3117.73it/s]

вопросов: 30
tool-вопросов: 6


## 1. RAG vs агент

Для каждого вопроса:

1. RAG: retrieve top-5 и один ответ;
2. агент: вызовы инструментов и финальный ответ;
3. judge оценивает оба ответа по correctness / faithfulness / citations.

Judge видит разный контекст: для RAG — top-5, для агента — то, что вернули
инструменты. Это честнее, чем оценивать агента по чужому retrieval.

30 × (2 ответа + 2 judge) плюс вызовы инструментов — ноутбук идёт несколько минут.

In [3]:
rows = []
started = time.perf_counter()


def safe_rag(question: str):
    last_error = None
    for _ in range(3):
        try:
            return pipeline.run(question)
        except Exception as exc:
            last_error = exc
            time.sleep(1.5)
    raise last_error


def safe_agent(question: str, thread_id: str) -> dict:
    last_error = None
    for _ in range(2):
        try:
            return run_agent(graph, question, thread_id)
        except Exception as exc:
            last_error = exc
            time.sleep(1.5)
    return {
        "answer": f"ERROR: {type(last_error).__name__}",
        "iterations": 0,
        "tool_sequence": [],
        "context": "",
        "n_sources": 0,
        "latency_ms": 0,
    }


for index, item in enumerate(items, start=1):
    question = item["question"]

    rag_started = time.perf_counter()
    try:
        rag = safe_rag(question)
        rag_answer = rag.answer
        rag_documents = rag.documents
    except Exception as exc:
        rag_answer = f"ERROR: {type(exc).__name__}"
        rag_documents = []
    rag_latency_ms = round((time.perf_counter() - rag_started) * 1000)

    agent = safe_agent(question, f"eval-{item['id']}")
    agent_context = agent["context"] or "Агент не вызвал инструменты."

    rag_context = format_context(rag_documents) if rag_documents else "Контекст не найден."
    rag_judge = judge_answer(
        question,
        item["reference_answer"],
        item["key_facts"],
        rag_context,
        rag_answer,
        llm=llm,
    )
    agent_judge = judge_answer(
        question,
        item["reference_answer"],
        item["key_facts"],
        agent_context,
        agent["answer"],
        llm=llm,
    )

    rows.append(
        {
            "id": item["id"],
            "topic": item["topic"],
            "type": item["type"],
            "difficulty": item["difficulty"],
            "question": question,
            "answer_rag": rag_answer,
            "answer_agent": agent["answer"],
            "judge_rag": rag_judge,
            "judge_agent": agent_judge,
            "citations_rag": citation_stats(rag_answer, len(rag_documents)),
            "citations_agent": citation_stats(
                agent["answer"], agent["n_sources"]
            ),
            "latency_rag_ms": rag_latency_ms,
            "latency_agent_ms": agent["latency_ms"],
            "iterations_agent": agent["iterations"],
            "tools_agent": agent["tool_sequence"],
            "n_sources_rag": len(rag_documents),
            "n_sources_agent": agent["n_sources"],
        }
    )

    elapsed = time.perf_counter() - started
    print(
        f"[{index}/{len(items)}] {item['id']} | "
        f"rag {rag_latency_ms}ms | agent {agent['latency_ms']}ms | "
        f"iter {agent['iterations']} | tools {agent['tool_sequence']} | "
        f"elapsed {elapsed:.0f}s"
    )
    time.sleep(0.15)

2026-11-09 00:09:31 [info     ] agent_step                     iteration=1 latency_ms=1202 tools=['search_wikipedia']


2026-11-09 00:09:36 [info     ] agent_step                     iteration=2 latency_ms=4958 tools=[]


[1/30] lipschitz-def-01 | rag 2919ms | agent 6185ms | iter 2 | tools ['search_wikipedia'] | elapsed 13s


2026-11-09 00:09:44 [info     ] agent_step                     iteration=1 latency_ms=2085 tools=['search_wikipedia', 'run_python']


2026-11-09 00:09:49 [info     ] agent_step                     iteration=2 latency_ms=4231 tools=[]


[2/30] lipschitz-example-02 | rag 2820ms | agent 6378ms | iter 2 | tools ['search_wikipedia', 'run_python'] | elapsed 25s


... (92 outputs left out)

2026-11-09 00:16:22 [info     ] agent_step                     iteration=1 latency_ms=1012 tools=['search_wikipedia', 'search_wikipedia']


2026-11-09 00:16:24 [info     ] agent_step                     iteration=2 latency_ms=2379 tools=['run_python']


2026-11-09 00:16:33 [info     ] agent_step                     iteration=3 latency_ms=8009 tools=[]


[30/30] multi-reduction-system-30 | rag 3221ms | agent 12001ms | iter 3 | tools ['search_wikipedia', 'search_wikipedia', 'run_python'] | elapsed 430s


## 2. Сводные метрики

In [4]:
def mean_judge(rows, key, field):
    values = [
        row[key][field]
        for row in rows
        if row.get(key) and row[key].get(field) is not None
    ]
    return round(sum(values) / len(values), 3) if values else None


def mean(values):
    values = list(values)
    return round(sum(values) / len(values), 1) if values else None


summary_rows = []
for mode in ["rag", "agent"]:
    judge_key = f"judge_{mode}"
    citation_key = f"citations_{mode}"
    latencies = sorted(row[f"latency_{mode}_ms"] for row in rows)
    summary_rows.append(
        {
            "mode": mode,
            "correctness": mean_judge(rows, judge_key, "correctness"),
            "faithfulness": mean_judge(rows, judge_key, "faithfulness"),
            "citations_score": mean_judge(rows, judge_key, "citations"),
            "has_citations_pct": round(
                100
                * sum(row[citation_key]["has_citations"] for row in rows)
                / len(rows),
                1,
            ),
            "valid_citation_ratio": round(
                mean(row[citation_key]["valid_ratio"] for row in rows), 3
            ),
            "latency_ms_mean": mean(latencies),
            "latency_ms_p50": latencies[len(latencies) // 2],
            "iterations_mean": round(
                sum(row["iterations_agent"] for row in rows) / len(rows), 2
            )
            if mode == "agent"
            else 1.0,
            "tool_calls_mean": round(
                sum(len(row["tools_agent"]) for row in rows) / len(rows), 2
            )
            if mode == "agent"
            else 0.0,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,mode,correctness,faithfulness,citations_score,has_citations_pct,valid_citation_ratio,latency_ms_mean,latency_ms_p50,iterations_mean,tool_calls_mean
0,rag,3.733,4.900,4.767,100.0,1.0,3051.4,3078,1.0,0.00
1,agent,4.633,4.533,4.600,100.0,1.0,7880.0,8055,2.4,2.03


## 3. По темам

In [5]:
topic_rows = []
for topic, group in pd.DataFrame(rows).groupby("topic", sort=True):
    topic_rows.append(
        {
            "topic": topic,
            "n": len(group),
            "correctness_rag": mean_judge(
                group.to_dict("records"), "judge_rag", "correctness"
            ),
            "correctness_agent": mean_judge(
                group.to_dict("records"), "judge_agent", "correctness"
            ),
            "latency_rag_ms": mean(group["latency_rag_ms"]),
            "latency_agent_ms": mean(group["latency_agent_ms"]),
        }
    )
topic_df = pd.DataFrame(topic_rows)
topic_df["delta_correctness"] = (
    topic_df["correctness_agent"] - topic_df["correctness_rag"]
).round(2)
topic_df

,topic,n,correctness_rag,correctness_agent,latency_rag_ms,latency_agent_ms,delta_correctness
0,integrating_factor,3,4.667,5.00,3137.7,6706.0,0.33
1,linear_ode,5,3.800,4.00,2345.4,9221.4,0.20
2,lipschitz,4,3.250,4.75,3375.8,7424.2,1.50
3,multi_hop,5,4.000,5.00,3717.2,9232.2,1.00
4,ode,4,3.750,5.00,2709.0,5472.2,1.25
5,picard,5,2.400,4.00,2985.4,7641.2,1.60
6,wronskian,4,4.750,5.00,3137.2,8555.5,0.25


## 4. Tool-choice

Прогоняем 6 вопросов из `tool_qa.jsonl` и сравниваем фактическую
последовательность инструментов с ожидаемой.

In [6]:
tool_rows = []
tool_started = time.perf_counter()

for index, item in enumerate(tool_items, start=1):
    run = safe_agent(item["question"], f"tool-eval-{item['id']}")
    tool_rows.append(
        {
            "id": item["id"],
            "topic": item["topic"],
            "difficulty": item["difficulty"],
            "question": item["question"],
            "expected_tools": item["expected_tools"],
            "used_tools": run["tool_sequence"],
            "iterations": run["iterations"],
            "latency_ms": run["latency_ms"],
            "answer": run["answer"],
        }
    )
    print(
        f"[{index}/{len(tool_items)}] {item['id']} | "
        f"expected {item['expected_tools']} | used {run['tool_sequence']} | "
        f"elapsed {time.perf_counter() - tool_started:.0f}s"
    )
    time.sleep(0.15)

tool_frame, tool_summary = evaluate_tool_questions(
    tool_items,
    {row["id"]: row["used_tools"] for row in tool_rows},
)
tool_frame = tool_frame.merge(
    pd.DataFrame(tool_rows)[["id", "latency_ms", "iterations"]],
    on="id",
    how="left",
)
tool_summary_df = pd.DataFrame([tool_summary])
tool_summary_df

2026-11-09 00:16:38 [info     ] agent_step                     iteration=1 latency_ms=781 tools=['run_python']


2026-11-09 00:16:40 [info     ] agent_step                     iteration=2 latency_ms=1207 tools=[]


[1/6] tool-charpoly-01 | expected ['run_python'] | used ['run_python'] | elapsed 3s


2026-11-09 00:16:41 [info     ] agent_step                     iteration=1 latency_ms=1017 tools=['run_python']


2026-11-09 00:16:43 [info     ] agent_step                     iteration=2 latency_ms=1968 tools=[]


[2/6] tool-wronskian-02 | expected ['run_python'] | used ['run_python'] | elapsed 6s


... (9 outputs left out)

2026-11-09 00:17:05 [info     ] agent_step                     iteration=1 latency_ms=1246 tools=['run_python']


2026-11-09 00:17:06 [info     ] agent_step                     iteration=2 latency_ms=1006 tools=[]


[6/6] tool-calc-06 | expected ['run_python'] | used ['run_python'] | elapsed 29s


,questions,tool_choice_accuracy,exact_tool_set_accuracy,tool_precision,tool_recall,tool_f1,unnecessary_tool_rate,sequence_accuracy
0,6,1.0,1.0,1.0,1.0,1.0,0.0,1.0


## 5. Сохранение

In [7]:
persisted_rows = [
    {
        key: value
        for key, value in row.items()
        if key not in {"answer_rag", "answer_agent"}
    }
    | {
        "answer_rag": row["answer_rag"][:1200],
        "answer_agent": row["answer_agent"][:1200],
    }
    for row in rows
]

payload = {
    "summary": summary_df.to_dict(orient="records"),
    "by_topic": topic_df.to_dict(orient="records"),
    "rows": persisted_rows,
    "tool_summary": tool_summary,
    "tool_rows": [
        {**row, "answer": row["answer"][:1200]} for row in tool_rows
    ],
}
output = Path("eval/results/agent_vs_rag_experiment.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("сохранено:", output)

сохранено: eval\results\agent_vs_rag_experiment.json


## 6. Как читать

- `correctness` — соответствие эталонному ответу и ключевым фактам;
- `faithfulness` — подтверждаются ли утверждения тем контекстом, который
  реально видела система;
- `citations_score` — корректность ссылок `[n]` по оценке judge;
- `latency_ms` — полное время ответа, включая retrieval и вызовы инструментов;
- `iterations_mean` — сколько раз агент обращался к LLM;
- `tool_choice_accuracy` — хотя бы один правильный инструмент;
- `exact_tool_set_accuracy` — набор инструментов совпал полностью;
- `sequence_accuracy` — правильный порядок для multi-tool.

RAG дешевле и предсказуемее. Агент выигрывает там, где нужно несколько
источников или вычисления, но платит latency и риском лишних вызовов.

## 7. Выводы

### Качество ответа

| Режим | correctness | faithfulness | citations | ответов со ссылками |
|---|---|---|---|---|
| RAG | 3.73 | **4.90** | **4.77** | 100% |
| агент | **4.63** | 4.53 | 4.60 | 100% |

Агент выиграл **+0.90** correctness, но проиграл по faithfulness (−0.37) и
citations (−0.17): его ответы длиннее, и judge находит больше утверждений,
которые не подтверждены найденными пассажами. Ссылки есть во всех ответах.

### Стоимость

| Режим | latency mean | latency p50 | iterations | tool calls |
|---|---|---|---|---|
| RAG | 3.1 с | 3.1 с | 1 | 0 |
| агент | 7.9 с | 8.1 с | 2.40 | 2.03 |

Агент в **2.6 раза медленнее**: в среднем 2 вызова инструментов и 2.4 итерации
LLM на вопрос.

### Correctness по темам

| Тема | RAG | агент | Δ |
|---|---|---|---|
| integrating factor | 4.67 | 5.00 | +0.33 |
| linear ODE | 3.80 | 4.00 | +0.20 |
| lipschitz | 3.25 | 4.75 | +1.50 |
| multi-hop | 4.00 | 5.00 | +1.00 |
| ODE | 3.75 | 5.00 | +1.25 |
| Picard | 2.40 | 4.00 | +1.60 |
| Wronskian | 4.75 | 5.00 | +0.25 |

### Tool-choice

| Метрика | Значение |
|---|---|
| вопросов | 6 |
| tool_choice_accuracy | 100% |
| exact_tool_set_accuracy | 100% |
| tool_precision / recall / F1 | 1.00 / 1.00 / 1.00 |
| sequence_accuracy | 100% |
| unnecessary_tool_rate | 0% |

Все 6 вопросов решены правильным набором инструментов и в правильном порядке.
Multi-tool вопросы: `search_wikipedia` → `run_python`, без лишних повторов.

### Где агент выиграл

Самый большой прирост — Picard (+1.6), Lipschitz (+1.5), ODE (+1.25) и
multi-hop (+1.0). Это вопросы, где одного top-5 недостаточно: агент делает
несколько поисков, получает больше пассажей и синтезирует ответ из разных
разделов. По вопросам: 11 улучшений, 18 совпадений, 1 регрессия.

### Где агент проиграл

Единственная регрессия — `linearode-charpoly-07`: correctness 5 → 4.
После исправления кодировки `run_python` повторов больше нет: максимум
3 итерации и 3 вызова инструментов, лимит не достигается.

Faithfulness ниже, потому что агент добавляет больше деталей, чем RAG.
Часть из них не подтверждается пассажами — это основной резерв качества.

### Вывод

Агент заметно лучше на сложных и multi-hop вопросах, но в 2.6 раза дороже по
latency и менее экономен в утверждениях. Для простых определений RAG
достаточно и он точнее по faithfulness.

Практичная схема — гибридная маршрутизация:

1. простой вопрос → RAG (быстро, дешево, высокая faithfulness);
2. multi-hop, несколько статей или вычисления → агент;
3. в ответе агента — ограничить детали тем, что подтверждено пассажами;
4. следующий шаг — измерить токены и стоимость, затем Docker/CI.
